In [1]:
from pathlib import Path
import operator
import cytoolz
import fastcluster

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from pyscenic.utils import load_motifs
from pyscenic.aucell import aucell
from pyscenic.binarization import binarize
from pyscenic.plotting import plot_binarization, plot_rss
from pyscenic.transform import df2regulons
import anndata

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, color_map='RdBu_r')

/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/home/dhy/.conda/envs/pyscenic5/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom 

scanpy==1.9.5 anndata==0.11.4 umap==0.5.4 numpy==1.26.1 scipy==1.11.3 pandas==2.1.1 scikit-learn==1.3.2 statsmodels==0.14.0 igraph==0.11.9 pynndescent==0.5.10


In [2]:

import numpy as np
import re

def sub(string,start=0,stop=-1) -> str:
	return string[start:stop]

def detect(pattern: str, string: str, flags=0) -> bool:
	return False if re.search(pattern=pattern,string=string,flags=flags) is None else True

def replace(pattern: str, repl: str, string: str, count:int = 0):
	return re.sub(pattern=pattern,repl=repl,string=string, count=count)

def grep(pattern: str, string: str, flags=0):
	bl = detect(pattern=pattern,string=string,flags=flags)
	return list(np.array(string)[bl])

def count(pattern: str, string: str, flags=0):
    return len(re.findall(pattern=pattern,string=string,flags=flags))
detects = np.vectorize(detect,otypes=[bool])

In [3]:


def select(frame, columns=None, pattern=None):
    """
    select a DataFrame columns according to `subsets` conditions
    """
    _objs = []
    if columns:
        cidx = frame.columns.isin(values=columns)
        _objs.append(frame.loc[:, cidx])
    if pattern:
        cidx = detects(string=frame.columns, pattern=pattern)
        _objs.append(frame.loc[:, cidx])
    return pd.concat(objs=_objs,axis=1)

In [4]:
def filter_regulons(motifs, db_names=("hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings",)):
    """
    从ctx.csv中筛选重要的regulons，之后再运行AUCell
    """
    motifs.columns = motifs.columns.droplevel(0)
    def contains(*elems):
        def f(context):
            return any(elem in context for elem in elems)
        return f
    # For the creation of regulons we only keep the 10-species databases and the activating modules. We also remove the
    # enriched motifs for the modules that were created using the method 'weight>50.0%' (because these modules are not part
    # of the default settings of modules_from_adjacencies anymore.
    lg = np.fromiter(map(cytoolz.compose(operator.not_, contains('weight>50.0%')), motifs.Context), dtype=bool) & \
        np.fromiter(map(contains(*db_names), motifs.Context), dtype=bool) & \
        np.fromiter(map(contains('activating'), motifs.Context), dtype=bool)
    motifs = motifs.loc[lg,:]

    # We build regulons only using enriched motifs with a NES of 3.0 or higher; we take only directly annotated TFs or TF annotated
    # for an orthologous gene into account; and we only keep regulons with at least 10 genes.
    regulons = list(filter(lambda r: len(r) >= 10, 
        df2regulons(motifs[(motifs.NES >= 3.0) 
        & ((motifs['Annotation'] == 'gene is directly annotated')
        | (motifs['Annotation'].str.startswith('gene is orthologous to')
            & motifs['Annotation'].str.endswith('which is directly annotated for motif')))
        ])))

    # Rename regulons, i.e. remove suffix.
    return list(map(lambda r: r.rename(r.transcription_factor), regulons))

#

In [5]:


random_state=2024
path_to_fast="/data/work/2026317luad/"

OUTPUT_DIR='/data/work/2026317luad/pyscenic/'

In [6]:
adata=sc.read_h5ad(path_to_fast + "GSE131907_GSE253013_raw.h5ad")

In [7]:



adata

print(adata.X.shape)  # 查看 adata.X 的形状
print(adata.var_names)  # 查看基因名称
print(adata.obs_names)  # 查看细胞名称



(21548, 25089)
Index(['RP11-34P13.7', 'FO538757.2', 'AP006222.2', 'RP4-669L17.10',
       'RP5-857K21.4', 'RP11-206L10.9', 'FAM87B', 'LINC00115', 'FAM41C',
       'RP11-54O7.1',
       ...
       'LINC00163', 'LINC00165', 'LINC00334', 'LINC00316', 'COL18A1-AS1',
       'AL592528.1', 'AP001476.2', 'AC136612.1', 'AC136616.1', 'AC007325.1'],
      dtype='object', length=25089)
Index(['g2_TAAGAGAGTTCGCTAA-25', 'g1_CGGGTCAAGCTGAACG-18',
       'g1_CGGCTAGGTTCGCTAA-18', 'g1_CGGCTAGAGGCGTACA-18',
       'g1_CGGCTAGAGAATAGGG-18', 'g1_CGGAGTCTCGCAAGCC-18',
       'g1_CGGAGTCGTTAAGATG-18', 'g1_CGGAGCTGTCTCTCTG-18',
       'g1_CGGACTGAGTAGCCGA-18', 'g1_CGGACGTTCTGGCGTG-18',
       ...
       'ACGATGTTCAAGGTAA_EBUS_28', 'AAATGCCCATTGGTAC_LUNG_T34',
       'TACGGGCAGCCTTGAT_EBUS_28', 'TCAGCAAAGGACGAAA_EBUS_28',
       'ACGATGTTCCAAGCCG_EBUS_06', 'TCAGCAAAGGTTACCT_EBUS_28',
       'TCACGAATCCGCATCT_LUNG_T18', 'AACTGGTGTCACTGGC_EBUS_28',
       'TAAGTGCTCCGAACGC_EBUS_28', 'AAGCCGCAGCTAAACA_EBUS_28'],

In [8]:



expression_matrix = pd.DataFrame(adata.X.toarray(), 
                                  index=adata.obs_names, 
                                  columns=adata.var_names)


In [9]:

df_motifs = load_motifs("/data/work/2026317luad/mEpi_cells_reg.csv")
regulons = filter_regulons(df_motifs)

regulons

FileNotFoundError: [Errno 2] No such file or directory: '/data/work/2026317luad/mEpi_cells_reg.csv'

In [ ]:

exp_mtx = expression_matrix.copy()

auc_mtx = aucell(exp_mtx=exp_mtx, signatures=regulons,seed=1314, num_workers=12) 

auc_mtx


In [ ]:


OUTPUT_DIR

auc_mtx.to_csv("/data/work/2026317luad/pyscenic/aucell.csv")

auc_mtx=pd.read_csv(OUTPUT_DIR+"aucell.csv", index_col=0)
bin_mtx, thresholds = binarize(auc_mtx,seed=1314,num_workers=12)
bin_mtx.to_csv("bin_mtx.csv")
thresholds.to_frame().rename(columns={0:'threshold'}).to_csv("thresholds.csv")


In [ ]:




# 假设你的两个 DataFrame 是 Ductal.obs 和 auc_mtx
filtered_auc_mtx = auc_mtx.loc[auc_mtx.index.intersection(adata.obs.index)]
#filtered_auc_mtx = filtered_auc_mtx.loc[:, (filtered_auc_mtx != 0).any(axis=0)]
filtered_auc_mtx

In [ ]:
import functools

remove = functools.partial(replace,repl='')
removes = np.vectorize(remove,otypes=[str])


bin_mtx.columns = removes(string=bin_mtx.columns, pattern=r'\(\+\)')
bin_mtx.columns 

filtered_bin_mtx=bin_mtx.loc[bin_mtx.index.isin(filtered_auc_mtx.index)]
#filtered_bin_mtx = filtered_bin_mtx.loc[:, (filtered_bin_mtx != 0).any(axis=0)]
#filtered_bin_mtx=filtered_bin_mtx.loc[filtered_bin_mtx.columns.intersection(filtered_auc_mtx.columns)]
common_columns = filtered_bin_mtx.columns.intersection(filtered_auc_mtx.columns)
filtered_bin_mtx = filtered_bin_mtx[common_columns]
filtered_bin_mtx


In [ ]:
def display_logos(df: pd.DataFrame,
                   top_target_genes: int = 3, 
                   base_url: str = "http://motifcollections.aertslab.org/v10nr_clust/logos/"
                   ,column_name_logo = "MotifLogo"
                   ,column_name_id  = "MotifID"
                   , column_name_target = "TargetGenes"
                   ):
    """
    :param df:
    :param base_url:
    """
    # Make sure the original dataframe is not altered.
    df = df.copy()
    
    # Add column with URLs to sequence logo.
    def create_url(motif_id):
        return '<img src="{}{}.png" style="max-height:124px;"></img>'.format(base_url, motif_id)
    df[("Enrichment", column_name_logo)] = list(map(create_url, df.index.get_level_values(column_name_id)))
    
    # Truncate TargetGenes.
    def truncate(col_val):
        return sorted(col_val, key=op.itemgetter(1))[:top_target_genes]
    df[("Enrichment", column_name_target)] = list(map(truncate, df[("Enrichment", column_name_target)]))
    
    MAX_COL_WIDTH = pd.get_option('display.max_colwidth')
    pd.set_option('display.max_colwidth', -1)
    display(HTML(df.head().to_html(escape=False)))
    pd.set_option('display.max_colwidth', MAX_COL_WIDTH)

def fetch_logo(regulon, base_url = "http://motifcollections.aertslab.org/v10nr_clust/logos/"):
    for elem in regulon.context:
        if elem.endswith('.png'):
            return '<img src="{}{}" style="max-height:124px;"></img>'.format(base_url, elem)
    return ""

auc_sum = auc_mtx.apply(sum,axis=0).sort_values(ascending=False)
fig, axes = plt.subplots(1, 5, figsize=(8, 2), dpi=100)
for x,y in enumerate(axes):
    plot_binarization(auc_mtx, auc_sum.index[x], thresholds[auc_sum.index[x]], ax=y)
plt.tight_layout()

In [ ]:

adata.obs

In [ ]:
cell_type_key = "Cell_type"


In [ ]:
import matplotlib.pyplot as plt

# 获取类别数量并生成调色板
n_categories = len(adata.obs[cell_type_key].cat.categories)
colors = plt.get_cmap('tab20').colors[:n_categories]

# 创建颜色字典，将每个类别映射到颜色
cell_type_color_lut = dict(zip(adata.obs[cell_type_key].cat.categories, colors))

# 将颜色字典保存到 adata.uns 中，供后续使用
adata.uns[f'{cell_type_key}_colors'] = [cell_type_color_lut[cat] for cat in adata.obs[cell_type_key].cat.categories]


cell_type_color_lut = dict(zip(adata.obs[cell_type_key].dtype.categories, adata.uns[f'{cell_type_key}_colors']))
cell_id2cell_type_lut = adata.obs[cell_type_key].to_dict()
bw_palette = sns.xkcd_palette(["white", "black"])

sns.set()
sns.set(font_scale=1.0)
sns.set_style("ticks", {"xtick.minor.size": 1, "ytick.minor.size": 0.1})
g = sns.clustermap(
    data=filtered_bin_mtx.T, 
    col_colors=filtered_bin_mtx.index.map(cell_id2cell_type_lut).map(cell_type_color_lut),
    cmap=bw_palette, figsize=(20,20)
    )
g.ax_heatmap.set_xticklabels([])
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_xlabel('Cells')
g.ax_heatmap.set_ylabel('Regulons')
g.ax_col_colors.set_yticks([0.5])
g.ax_col_colors.set_yticklabels(['Cell_type2'])
g.cax.set_visible(False)

In [ ]:
df_regulons = pd.DataFrame(data=[list(map(operator.attrgetter('name'), regulons)),
    list(map(len, regulons)),
    list(map(fetch_logo, regulons))], 
    index=['name', 'count', 'logo']).T



df_regulons

In [ ]:
#	hocomoco__NR1H4_HUMAN.H11MO.1.B.png

MAX_COL_WIDTH = pd.get_option('display.max_colwidth')
pd.set_option('display.max_colwidth', None)
import IPython.display
IPython.display.display(IPython.display.HTML(df_regulons[df_regulons['name'].isin(['CUX1', 
                                                                                   'EGR1', 
                                                                                   'ETV1',
                                                                                  'ETV5', 
                                                                                   'TBX1', 
                                                                                   'ZNF569',
                                                                                   
                                                                                  
                                                                                  ])].to_html(escape=False)))
pd.set_option('display.max_colwidth', MAX_COL_WIDTH)

In [ ]:
#细胞特异 REGULATORS

from pyscenic.export import add_scenic_metadata
add_scenic_metadata(adata, filtered_bin_mtx, regulons);

df_scores=select(adata.obs,columns=[cell_type_key],pattern=r'^Regulon\(')
df_results = ((df_scores.groupby(by=cell_type_key).mean() - 
               df_scores[df_scores.columns[1:]].mean())/ df_scores[df_scores.columns[1:]].std()).stack().reset_index().rename(columns={'level_1': 'regulon', 0:'Z'})
df_results['regulon'] = list(map(lambda s: s[8:-1], df_results.regulon))
df_results[(df_results.Z >= 0.001)].sort_values('Z', ascending=False).head()

In [ ]:
df_heatmap = pd.pivot_table(data=df_results[df_results.Z >= 0.3].sort_values('Z', ascending=False),
                            index=cell_type_key, columns='regulon', values='Z')
fig, ax1 = plt.subplots(1, 1, figsize=(14, 14))
sns.heatmap(df_heatmap, ax=ax1,
            annot=True, fmt=".1f", linewidths=.6, 
            cbar=False, square=True, linecolor='gray',cmap="viridis", annot_kws={"size": 8})
ax1.set_ylabel('');
plt.savefig("/data/work/2026317luad/pyscenic/scenic_heatmap02.pdf", dpi=600, bbox_inches='tight')

In [ ]:
auc_mtx

In [ ]:
from pyscenic.rss import regulon_specificity_scores
rss = regulon_specificity_scores(auc_mtx, adata.obs[cell_type_key])
rss

In [ ]:
import matplotlib.pyplot as plt

# 遍历所有细胞类型
for cell in adata.obs[cell_type_key].unique():
    plot_rss(rss, cell_type=cell, top_n=5)
    # 使用细胞类型作为文件名的一部分保存图像
    plt.savefig(f"/data/work/2026317luad/pyscenic/pyscenic_{cell}02.pdf", dpi=600, bbox_inches='tight')
    plt.clf()  # 清除当前图像，防止多个图叠加